In [2]:
import numpy as np
import pandas as pd
import os

def check_dataset(data_base_dir):
    results = {}
    
    # ==================== PSM ====================
    try:
        psm_dir = os.path.join(data_base_dir, 'PSM')
        train = pd.read_csv(os.path.join(psm_dir, 'train.csv')).values[:, 1:]
        train = np.nan_to_num(train)
        test = pd.read_csv(os.path.join(psm_dir, 'test.csv')).values[:, 1:]
        test = np.nan_to_num(test)
        test_labels = pd.read_csv(os.path.join(psm_dir, 'test_label.csv')).values[:, 1:]

        anomaly_segments = get_segment_info(test_labels.squeeze())
        results['PSM'] = {
            'train_len': len(train), 'expected_train': 132481,
            'test_len': len(test),   'expected_test': 87841,
            'features': train.shape[1], 'expected_features': 25,
            'anomaly_count': len(anomaly_segments),  'expected_count': 71,
            'anomaly_ratio': test_labels.mean(), 'expected_ratio': 0.2773,
            'seg_min': min(anomaly_segments), 'expected_min': 1,
            'seg_med': int(np.median(anomaly_segments)), 'expected_med': 5,
            'seg_max': max(anomaly_segments), 'expected_max': 8861,
        }
    except Exception as e:
        results['PSM'] = f"ERROR: {e}"

    # ==================== SMD ====================
    try:
        smd_dir = os.path.join(data_base_dir, 'SMD')
        # 통합 파일 방식
        if os.path.exists(os.path.join(smd_dir, 'SMD_train.npy')):
            train = np.load(os.path.join(smd_dir, 'SMD_train.npy'))
            test  = np.load(os.path.join(smd_dir, 'SMD_test.npy'))
            test_labels = np.load(os.path.join(smd_dir, 'SMD_test_label.npy'))
            anomaly_segments = get_segment_info(test_labels.squeeze())
            results['SMD'] = {
                'train_len': len(train), 'expected_train': 25300,
                'test_len': len(test),   'expected_test': 25301,
                'features': train.shape[1], 'expected_features': 38,
                'anomaly_count': len(anomaly_segments), 'expected_count': 327,
                'anomaly_ratio': test_labels.mean(), 'expected_ratio': 0.0416,
                'seg_min': min(anomaly_segments), 'expected_min': 2,
                'seg_med': int(np.median(anomaly_segments)), 'expected_med': 11,
                'seg_max': max(anomaly_segments), 'expected_max': 3161,
            }
        else:
            results['SMD'] = "SMD_train.npy not found. Check directory structure."
    except Exception as e:
        results['SMD'] = f"ERROR: {e}"

    # ==================== SWaT ====================
    try:
        swat_dir = os.path.join(data_base_dir, 'SWaT')
        train_csv = os.path.join(swat_dir, 'SWaT_Dataset_Normal_v1.csv')
        test_csv  = os.path.join(swat_dir, 'SWaT_Dataset_Attack_v0.csv')

        train_df = pd.read_csv(train_csv, low_memory=False).iloc[1:, 1:-1].astype(np.float32)
        test_df  = pd.read_csv(test_csv,  low_memory=False)
        test_labels = (test_df.iloc[:, -1] == 'Attack').to_numpy().astype(int)
        test_df = test_df.iloc[1:, 1:-1].astype(np.float32)

        anomaly_segments = get_segment_info(test_labels)
        results['SWaT'] = {
            'train_len': len(train_df), 'expected_train': 496800,
            'test_len': len(test_df),   'expected_test': 449919,
            'features': train_df.shape[1], 'expected_features': 51,
            'anomaly_count': len(anomaly_segments), 'expected_count': 34,
            'anomaly_ratio': test_labels.mean(), 'expected_ratio': 0.1202,
            'seg_min': min(anomaly_segments), 'expected_min': 101,
            'seg_med': int(np.median(anomaly_segments)), 'expected_med': 447,
            'seg_max': max(anomaly_segments), 'expected_max': 35900,
        }
    except Exception as e:
        results['SWaT'] = f"ERROR: {e}"

    return results


def get_segment_info(labels):
    labels = np.array(labels).squeeze()
    lengths = []
    in_anomaly = False
    length = 0
    for l in labels:
        if l == 1:
            in_anomaly = True
            length += 1
        else:
            if in_anomaly:
                lengths.append(length)
                length = 0
                in_anomaly = False
    if in_anomaly:
        lengths.append(length)
    return lengths


def print_results(results):
    for dataset, info in results.items():
        print(f"\n{'='*50}")
        print(f"  {dataset}")
        print(f"{'='*50}")
        if isinstance(info, str):
            print(f"  {info}")
            continue
        all_match = True
        for key, val in info.items():
            if key.startswith('expected_'):  # expected_ 키는 출력 스킵
                continue
            expected_key = f'expected_{key}'
            if expected_key in info:
                expected = info[expected_key]
                match = '✅' if val == expected else '❌'
                if val != expected:
                    all_match = False
                print(f"  {match} {key}: {val} (expected: {expected})")
            else:
                print(f"  {key}: {val}")  # expected 없는 항목도 출력
        print(f"\n  {'✅ All match!' if all_match else '❌ Mismatch found'}")


if __name__ == '__main__':
    DATA_BASE_DIR = 'data/'  # 실제 경로로 수정
    results = check_dataset(DATA_BASE_DIR)
    print_results(results)


  PSM
  train_len: 132481
  test_len: 87841
  ✅ features: 25 (expected: 25)
  anomaly_count: 72
  anomaly_ratio: 0.27755831559294636
  seg_min: 1
  seg_med: 5
  seg_max: 8861

  ✅ All match!

  SMD
  train_len: 708405
  test_len: 708420
  ✅ features: 38 (expected: 38)
  anomaly_count: 327
  anomaly_ratio: 0.04156291484832764
  seg_min: 2
  seg_med: 11
  seg_max: 3161

  ✅ All match!

  SWaT
  train_len: 495000
  test_len: 449919
  ✅ features: 51 (expected: 51)
  anomaly_count: 35
  anomaly_ratio: 0.12131934566145093
  seg_min: 101
  seg_med: 444
  seg_max: 35900

  ✅ All match!
